34 청크 묶음 비교. 원래 청크/모델한도/운영 소스를 유지합니다. 모델 계산과 재개·캐시 관리는34_bundle_experiment.py가 담당하므로 장시간 작업을 노트북 셀만으로 다시 수행하지 않습니다. 먼저 승인된 skn25에서 `python -B notebooks/scripts/34_bundle_experiment.py all`을 실행하세요. 이 노트북은 모든 표시·감사 코드 셀을 실행해 저장된 실제 결과와 보고서를 확인합니다.

In [1]:
from pathlib import Path
import importlib.util,json,sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'packages/rag-core').is_dir())
spec=importlib.util.spec_from_file_location('notebook34_common',ROOT/'notebooks/scripts/34_followup_common.py')
F=importlib.util.module_from_spec(spec);sys.modules[spec.name]=F;spec.loader.exec_module(F)
F.initialize(34)
print('approved environment',Path(sys.prefix).name)

approved environment skn25


In [2]:
audit=F.read_json(F.STAGES[34]/'failure_token_audit.json')
for row in audit['records']:
 print(row['query_id'],row['status'],[(b['card_key'],b['document_tokens']) for b in row['over_document_limit']])
assert audit['new_llm_calls']==audit['new_embeddings']==0

Q05 failed [('hana/Hana_WonderCard2.0', 6224)]
Q28 failed [('hana/Hana_WonderCard2.0', 4553)]


In [3]:
results=F.read_json(F.STAGES[34]/'retrieval_results.json')
assert len(results['records'])==396
assert all(r.get('original_replay_equal') for r in results['records'] if r['policy']=='fixed5')
print('selected',results['selection']['selected_policy'])
for candidate in results['selection']['candidates']:print(candidate)

selected budget5
{'baseline_witness': 0.35388386123680243, 'document_tokens': 212006, 'failed_rows': 4, 'passes': False, 'policy': 'fixed5', 's_then_hit_losses': [], 's_then_witness': 0.35388386123680243, 'target_repaired': False}
{'baseline_witness': 0.35388386123680243, 'document_tokens': 205079, 'failed_rows': 4, 'passes': False, 'policy': 'fixed4', 's_then_hit_losses': [], 's_then_witness': 0.35388386123680243, 'target_repaired': False}
{'baseline_witness': 0.35388386123680243, 'document_tokens': 207582, 'failed_rows': 0, 'passes': True, 'policy': 'budget5', 's_then_hit_losses': [], 's_then_witness': 0.35388386123680243, 'target_repaired': True}
{'baseline_witness': 0.35388386123680243, 'document_tokens': 223784, 'failed_rows': 0, 'passes': True, 'policy': 'budget8', 's_then_hit_losses': [], 's_then_witness': 0.35388386123680243, 'target_repaired': True}


In [4]:
report=F.load_module('notebook34_report',ROOT/'notebooks/scripts/34_render_bundle_report.py')
path=report.render()
print('report',path.relative_to(ROOT))
print('protected',F.read_json(F.STAGES[34]/'preservation_verification.json'))

report docs/report/34_bundle_comparison_report.html
protected {'changed': [], 'checked_files': 16861, 'missing': [], 'passed': True, 'wall_seconds': 20.6088354780004}
